## Plotting with SCAN Data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression



In [ ]:
def plot_nutrient_data(df, start_date, end_date, param, ax, check_valid = True, **kwargs):
    df.loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = param, **kwargs)


def plot_nutrient_data_by_bottle(df, start_date, end_date, param, ax, check_valid = True, **kwargs):
    first_rep = df['Bottle Replicate'] == 1
    print(first_rep)
    df[first_rep].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = 'First Bottle',  **kwargs)
    df[~first_rep].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', color = 'r', rot=45, label = 'Second Bottle',  **kwargs)


def plot_nutrient_data_by_rundate(df, start_date, end_date, param, ax, check_valid=True, **kwargs):
    rundates = df['AA500 Run Date'].unique()
    from itertools import cycle
    color_cycler = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = cycle(color_cycler)
    for date, color in zip(rundates, colors):
        df[df['AA500 Run Date'] == date].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, color=color, label = 'Run on: ' + date.strftime('%m/%d/%Y'),  **kwargs)
    

def plot_nutrient_data_by_type(df, start_date, end_date, param, ax, check_valid=True, **kwargs):
    sampletypes = df['Sample Type'].unique()
    from itertools import cycle
    color_cycler = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = cycle(color_cycler)
    for type, color in zip(sampletypes, colors):
        df[df['Sample Type'] == type].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, color=color, label = type,  **kwargs)
    

# Import Data

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/'
rme_results = pd.read_csv(result_dir+'rme_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
rme_results

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True)
rme_cleaned_uncorrected = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected.csv', index_col='Date/Time', parse_dates=True)
rme_cleaned['second_derivative_no3_mgl_bias_correct'] = rme_cleaned['second_derivative_no3_mgl_correct']-.1138
rme_cleaned_uncorrected['second_derivative_no3_mgl_bias_correct'] = rme_cleaned_uncorrected['second_derivative_no3_mgl_correct']-.1138

scan_data = rme_cleaned[np.arange(220,735, 2.5).astype(str)]


In [ ]:
rme_cleaned_uncorrected

In [ ]:
rme_results

In [ ]:
nitrate_merged = pd.merge_asof(rme_results, scan_data, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('4h')).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
nitrate_merged

In [ ]:
nitrate_merged_uncorrected = pd.merge_asof(rme_results, rme_cleaned_uncorrected[np.arange(220,735, 2.5).astype(str)], left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('4h')).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
nitrate_merged_uncorrected

# PLSR

## Turbidity Corrected

In [ ]:
scaler = StandardScaler()
scan_data_scaled = scaler.fit_transform(scan_data)

# Fit PCA
pca = PCA()
principal_components = pca.fit_transform(scan_data_scaled)
pca_results = pd.DataFrame()
pca_results['Explained Variance'] = pca.explained_variance_ratio_
pca_results['Cumulative Explained Variance'] = pca_results['Explained Variance'].cumsum()
pca_results.index = pca_results.index + 1

fig, ax = plt.subplots()
pca_results.plot(y='Cumulative Explained Variance', ylabel = 'Cumulative Explained Variance', xlabel = 'Principal Components', xlim=(0,10), ax=ax)
ax.axhline(y=.95, color = 'r', linestyle = '--')


- 4 components explains 95% of variance

In [ ]:
# Create a DataFrame for the principal components

pca = PCA(n_components=2)
principal_components = pca.fit_transform(scan_data_scaled)
pc_df = pd.DataFrame(principal_components, index=scan_data.index, columns=['PC1', 'PC2'])


# Plot the first two principal components
plt.figure(figsize=(10,6))
plt.scatter(pc_df['PC1'], pc_df['PC2'], alpha=0.7)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of SCAN Data')
plt.grid(True)
plt.show()

# Explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
X = nitrate_merged[np.arange(220.0, 735.0, 2.5).astype(str)]
Y = nitrate_merged['Nitrate mean']

pls4 = PLSRegression(n_components=4, scale=False)
pls4.fit(X, Y)

Y_pred = pls4.predict(rme_cleaned[np.arange(220.0, 735.0, 2.5).astype(str)])
rme_cleaned['PLSR'] = Y_pred

In [ ]:
len(Y_pred)

## Uncorrected

In [ ]:
scaler = StandardScaler()
scan_data_scaled = scaler.fit_transform(rme_cleaned_uncorrected[np.arange(220,735, 2.5).astype(str)])

# Fit PCA
pca = PCA()
principal_components = pca.fit_transform(scan_data_scaled)
pca_results = pd.DataFrame()
pca_results['Explained Variance'] = pca.explained_variance_ratio_
pca_results['Cumulative Explained Variance'] = pca_results['Explained Variance'].cumsum()
pca_results.index = pca_results.index + 1

fig, ax = plt.subplots()
pca_results.plot(y='Cumulative Explained Variance', ylabel = 'Cumulative Explained Variance', xlabel = 'Principal Components', xlim=(0,5), ax=ax)
ax.axhline(y=.95, color = 'r', linestyle = '--')


- 2 components explains 95% of variance for uncorrected S::CAN data

In [ ]:
# Create a DataFrame for the principal components

pca = PCA(n_components=2)
principal_components = pca.fit_transform(scan_data_scaled)
pc_df = pd.DataFrame(principal_components, index=scan_data.index, columns=['PC1', 'PC2'])


# Plot the first two principal components
plt.figure(figsize=(10,6))
plt.scatter(pc_df['PC1'], pc_df['PC2'], alpha=0.7)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of SCAN Data')
plt.grid(True)
plt.show()

# Explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
scan_data_scaled

In [ ]:
X = nitrate_merged_uncorrected[np.arange(220.0, 735.0, 2.5).astype(str)]
Y = nitrate_merged_uncorrected['Nitrate mean']

pls4 = PLSRegression(n_components=4, scale=False)
pls4.fit(X, Y)

Y_pred = pls4.predict(rme_cleaned_uncorrected[np.arange(220.0, 735.0, 2.5).astype(str)])
rme_cleaned['PLSR Uncorrected'] = Y_pred

In [ ]:
fig, ax= plt.subplots(figsize=(15, 9), nrows=2, sharex=True)
plot_nutrient_data_by_bottle(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[0])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct', 'PLSR', 'PLSR Uncorrected'], color=['g', 'y','b','c'], ax=ax[0], rot=45, title='All Data')

plot_nutrient_data_by_bottle(rme_results[rme_results['Nitrate QA']==''], '02/01/25', '11/01/25', 'Nitrate', ax[1])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct', 'PLSR', 'PLSR Uncorrected'], color=['g', 'y','b','c'], ax=ax[1], rot=45, title='Excluding Flagged Data')

fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[0], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[1], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')
rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['second_derivative'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:orange', secondary_y=True)


plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[2], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[3], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[4], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['PLSR Uncorrected'], ax=ax[4], rot=45, title='Uncorrected PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[0])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[1])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[2])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[3])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[4])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['PLSR Uncorrected'], ax=ax[4], rot=45, title='Uncorrected PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['03-15-2025':'05-01-2025'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
rme_cleaned.columns

## Correlation Plots

In [ ]:
def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='b')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]))
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]))
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2))
    return coeff, r2

In [ ]:
rme_merged_clean = pd.merge_asof(rme_results, rme_cleaned, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h')).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct'])
rme_merged_clean = rme_merged_clean.drop(['2/15/25 3:30:00','2-22-25 11:00']).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct']) # drop two outliers from timeseries


In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(11,11))

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .2, .02, ax[0,0])

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Corrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, 2, .3, ax[1,0])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Uncorrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[0,1])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .03, ax[1,1])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,2], title='Uncorrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[0,2])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,2], title='Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[1,2])

rme_merged_clean.plot(x='Nitrate mean', y='PLSR', xerr='Nitrate err',kind='scatter', ax=ax[2,0], title='Corrected PLSR')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'PLSR', .3, .3, .03, ax[2,0])

rme_merged_clean.plot(x='Nitrate mean', y='PLSR Uncorrected', xerr='Nitrate err',kind='scatter', ax=ax[2,1], title='Uncorrected PLSR')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'PLSR Uncorrected', .3, .3, .03, ax[2,1])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_bias_correct', xerr='Nitrate err',kind='scatter', ax=ax[2,2], title='Bias Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_bias_correct', .3, .3, .03, ax[2,2])

for a in ax.flatten():
    a.set_xlim(0, .6)
    a.set_ylim(0, .6)

ax.flatten()[3].set_ylim(0, 7)
fig.suptitle('RME Calibration Plots')
fig.tight_layout()


In [ ]:
rme_merged_clean

In [ ]:
rme_results.plot(y='Nitrate mean', kind='hist', xlabel = 'Nitrate Concentration (mg/L)')

## Plotting Other Nutrients

In [ ]:
rme_results_copy = rme_results
rme_results_copy['N-P Ratio mean'] = rme_results_copy['Nitrate mean']/rme_results_copy['Phosphate mean']
rme_results_copy['N-NH3 Ratio mean'] = rme_results_copy['Nitrate mean']/rme_results_copy['Ammonium mean']
rme_results_copy['N-P Ratio err'] = 0
rme_results_copy['N-NH3 Ratio err'] = 0


In [ ]:
fig, ax= plt.subplots(figsize=(15, 9),nrows=2,  sharex=True)
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Nitrate', ax[0], color='green', title= 'Concentration')
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Phosphate',  ax[0],color='red')
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Ammonium', ax[0], color='blue' )

plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'N-P Ratio', ax[1], color='red', title= 'Ratio', logy=True)
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'N-NH3 Ratio', ax[1], color='blue', title= 'Ratio', logy=True)

ax[0].set_ylabel('Concentration (mg/L)')
ax[1].set_ylabel('Concentration Ratio')
ax[1].axhline(1, color='k', ls='--')


fig.tight_layout()

In [ ]:
fieldblank_results = pd.read_csv(result_dir+'field_blank_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)